# Quantify invasion

**Purpose.** Classify pathogens as intracellular or extracellular using differential-staining measurements and summarize invasion efficiency.

**Recommended use.** Use for invasion or attachment assays with an extracellular marker.

**Primary outputs.** Per-object invasion labels and per-well or per-condition invasion estimates.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.submodules.analyze_invasion`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.analyze_invasion)

```python
analyze_invasion(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.submodules import analyze_invasion

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.submodules.analyze_invasion`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.analyze_invasion)


#### Assay Inputs

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`parasite_table`** *(optional)* — (str) - Table in measurements/measurements.db holding one row per segmented parasite. It is read directly rather than through the usual merge, because that merge collapses pathogen rows onto their host cell and would sum several parasites' stain intensities into a single row. Change it only if measure_crop wrote the parasite objects under a non-standard name. Default 'pathogen'.
- **`compartment`** *(optional)* — (str) - Prefix used by per-object measurement columns, so 'pathogen' selects pathogen_area and pathogen_channel_1_percentile_95. It must match the object type contained in the table; otherwise the run stops and reports the unresolved area and intensity columns. Default 'pathogen'.

#### Channels & Intensity

- **`outside_channel`** *(required)* — (int) - Zero-indexed channel of the pre-permeabilisation antibody, which labels parasites remaining outside the host cell. Classification thresholds this channel, so an incorrect index changes the assay readout to the signal measured in another channel without raising an error. This is an image-channel index, not measure.py's &lt;object&gt;_channel_&lt;n&gt;_outside_* columns, which quantify the ring outside an object's mask. Default 1.
- **`total_channel`** *(required)* — (int or None) - Zero-indexed channel of the post-permeabilisation antibody that stains every parasite. Nothing is classified from it; it only supplies the intensity that min_total_intensity filters on, so an incorrect value costs nothing until that filter is switched on. Default 0.
- **`intensity_statistic`** *(optional)* — (str) - Per-object statistic of the pre-permeabilisation channel used for thresholding. Because the stain is localized to the parasite surface, the object mean divides rim signal by the full area and can classify a larger parasite as dimmer than a smaller parasite with equivalent surface staining. A percentile of rim-pixel intensity reduces this size-dependent bias. Default 'mean'.
- **`background_correction`** *(optional)* — (str) - Per-object local background subtracted from the outside-stain statistic before thresholding. 'auto' uses the median of the five-pixel ring outside the parasite mask, removing a per-field offset without a flat-field image; 'none' performs no subtraction. A brightly stained attached parasite may produce an antibody halo that extends into the reference ring, in which case subtraction can reduce the signal required for classification. Default 'none'.
- **`min_total_intensity`** *(optional)* — (float or None) - Minimum mean intensity in the post-permeabilisation channel for an object to count as a parasite at all. That antibody stains every parasite, so an object dark in it is debris inside the pathogen mask rather than a dim parasite, and it would otherwise contribute a background-level outside signal and be scored invaded. None applies no filter. Default None.

#### Thresholding

- **`outside_threshold_method`** *(optional)* — (str) - Method used to derive the outside-stain threshold from each field when neither a fixed value nor control wells are supplied: 'otsu', 'triangle', 'li', 'yen', or 'mean'. These methods identify a partition but do not test whether the distribution is bimodal; that evaluation is performed separately. 'triangle' is appropriate for a strongly skewed distribution with a small stained minority, whereas 'otsu' is appropriate for a more balanced distribution. Default 'otsu'.
- **`outside_threshold`** *(optional)* — (float or None) - Fixed threshold for the outside-stain statistic, applied to every field and overriding both the automatic method and control wells. Use only after independent calibration: a value above the true threshold misclassifies attached parasites as invaded and inflates invasion efficiency. Control wells, when supplied, remain the reference against which quality control evaluates the fixed value. None derives the threshold per field. Default None.
- **`threshold_agreement_tolerance`** *(optional)* — (float) - Relative distance a threshold may sit from its reference before the field and well are flagged; the reference is the control-derived cut when controls exist, otherwise the field's own automatic cut. 0.5 means a factor of two. Lower it to catch smaller drifts between a fixed threshold and what the data would have chosen. Default 0.5.
- **`threshold_sensitivity`** *(optional)* — (float) - Fractional amount the threshold is moved up and down to produce the invasion_efficiency_low_threshold and _high_threshold bracket, which shows how much of a well's answer is the threshold rather than the biology. Widening it widens the bracket and makes the inflation flag more eager. Default 0.25.
- **`bimodality_cutoff`** *(optional)* — (float) - Minimum bimodality coefficient required for an unflagged field- or well-level efficiency estimate. Values below the threshold are retained but flagged because the intensity distribution provides insufficient evidence for two populations. Increasing the threshold requires stronger bimodality. Default 0.5555555555555556.
- **`extracellular_class`** *(optional)* — (str) - Classification policy for parasites that overlap no host cell. 'attached' assigns them to the attached class independent of stain intensity because they cannot be intracellular; 'classify' uses the stain signal when host-cell segmentation is uncertain; 'exclude' removes them before summary calculations. n_no_host_cell reports their count under every policy. Default 'attached'.

#### Controls & Minimum Counts

- **`control_wells`** *(conditionally required)* — (list or None) - Wells whose parasites carry no pre-permeabilisation stain, providing the empirical negative distribution used to set the threshold. Specify a column ('c12'), row ('r1'), well ('r1_c12'), or complete plate key. These staining-control wells are excluded from efficiency calculations because they are not experimental conditions. None uses the automatic per-field method. Screen regression uses this key for a separate purpose, where it must be a list matching filter_value. Default None.
- **`control_quantile`** *(optional)* — (float) - Quantile of the control wells' outside-stain distribution used as the threshold. A value of 0.99 classifies approximately one percent of genuinely unstained parasites as attached. Lowering it toward 0.95 reduces false invaded classifications while increasing false attached classifications; raising it has the opposite effect. Default 0.99.
- **`min_control_objects`** *(optional)* — (int) - Minimum number of objects required from a plate's control wells before their quantile is used as a threshold. Below this value, the plate uses the automatic per-field method and records the fallback rather than estimating a 99th percentile from an insufficient sample. Default 10.
- **`min_objects_for_threshold`** *(optional)* — (int) - Minimum number of objects required to derive a field-specific threshold. Below this count, the field uses its well threshold and then its plate threshold; automatic_source records the level used. Increasing the value improves statistical stability but reduces adaptation to local illumination variation. Default 10.
- **`min_objects_for_bimodality`** *(optional)* — (int) - Objects required before the bimodality coefficient is computed at all; below it the coefficient is left NaN and the field or well is flagged. The statistic exceeds its cutoff on genuinely unimodal data about 45% of the time at ten objects and 15% at twenty, so computing it there would silence the check exactly where the classification is least trustworthy. Default 30, where that false-pass rate is 5%.
- **`min_parasites_per_well`** *(optional)* — (int) - Minimum number of scored parasites required for an unflagged well-level efficiency estimate. At n=50 and p=0.5, the normal-approximation 95% interval has an approximate half-width of 13.9 percentage points. Estimates below the threshold remain in the output with a flag and their n_total value. Default 50.
- **`inflation_warn`** *(optional)* — (float) - Additional invasion efficiency, in proportion units, that increasing the threshold by threshold_sensitivity may add to a well before the well is flagged. Only the upward change is monitored because decreasing the threshold can only reclassify invaded parasites as attached and cannot create a positive invasion result. A value of 0.05 flags a well whose efficiency would increase by more than five percentage points. Default 0.05.

#### Object Filtering

- **`min_parasite_area`** *(optional)* — (int or float) - Smallest object area in pixels retained as a parasite. Smaller objects are treated as debris because their outside-stain statistic is estimated from too few pixels for stable thresholding. Increase it when pathogen masks are over-segmented into small fragments. Default 0, which applies no area filter.
- **`max_parasite_area`** *(optional)* — (float or None) - Largest object area in pixels kept as a parasite. Anything bigger is several parasites merged by the mask, whose rim statistic mixes them and whose single classification then stands for all of them. None keeps everything. Default None.

#### Condition Metadata

- **`cell_types`** *(optional)* — (list) - Names of the host cell lines in the experiment, e.g. ['HeLa']. Each name is written into the host_cells column and folded into the combined condition label used for grouping and plotting; the list is positionally paired with cell_plate_metadata, which says which wells hold each one. Default ['HeLa'].
- **`cell_plate_metadata`** *(optional)* — (list of lists) - Wells occupied by each entry of cell_types, with one inner list per cell type in the same order, for example [['c2','c3'],['c4']]. Each identifier must start with 'c' (column) or 'r' (row); invalid identifiers are skipped without an exception and those wells receive no host_cells label. Because 'condition' combines the labels that are present, a typographical error changes the comparison without raising an error. Default None.
- **`pathogen_types`** *(required)* — (list) - Names given to each pathogen condition on the plate, e.g. ['wt','ku80']. Element i is written into the pathogen column for every well listed in pathogen_plate_metadata[i] and folded into the combined condition label used for grouping and plotting. Must match pathogen_plate_metadata in length and order; None skips pathogen annotation. Default ['pathogen_1', 'pathogen_2'] for the dataset builders, ['pc'] for the control-based paths, None where types are not used.
- **`pathogen_plate_metadata`** *(required)* — (list of lists) - Well locations of each pathogen condition, one inner list per entry in pathogen_types. Every item must be a row or column ID string such as 'c1' or 'r3'; anything else is silently ignored and those wells stay unannotated. Ranges like 'c2-c11' are not expanded - list each row/column. Do not leave it None while pathogen_types is set: annotation is not skipped, every row is labelled with the first pathogen_types entry. Defaults: None in the plot-from-db settings, [['c1','c2','c3'],['c4','c5','c6']] for recruitment analysis.
- **`treatments`** *(optional)* — (list) - Names of the drug or treatment conditions in the experiment, e.g. ['dmso','lovastatin']. Each name is written into the treatment column and folded into the combined condition label used for grouping and plotting; positionally paired with treatment_plate_metadata (or treatment_loc), which lists the wells for each. Default ['cm','lovastatin'].
- **`treatment_plate_metadata`** *(optional)* — (list of lists) - Wells that received each treatment, with one inner list per treatment in the same order, for example [['r1','r2','r3'],['r4','r5','r6']]. Entries must start with 'r' (row) or 'c' (column); other entries are ignored and receive no treatment label. Unlisted wells remain in the output, and their condition values contain only the available cell, pathogen, or treatment labels. Default None.
- **`group_column`** *(optional)* — (str) - Column whose values become the experimental conditions compared against each other; 'condition' is the combined host-cell / pathogen / treatment label built from the plate-metadata maps. Point it at 'pathogen' or 'treatment' to compare on one factor alone. Rows with no value here are dropped before anything is counted. Default 'condition'.
- **`level`** *(optional)* — (str) - Result level. For regression, 'both' writes results_grna.csv and results_gene.csv and corrects each family separately; 'grna' reports guide effects, and 'gene' pools guides by gene. Nonparametric inference also honours this choice. Mixed models disable it because they estimate gene effects with guides nested inside genes. For proportion plots, the same key selects 'object', 'well', or 'plate' aggregation. Default 'both' for regression and 'object' for proportions.
- **`change_plate`** *(optional)* — (bool) - Relabel each source directory as plate1, plate2, ... instead of trusting the plate ID stored in its database. Use it when several plates were written under the same name, which would otherwise let two plates' fields pool into one threshold and one well. Default False.

#### Assay Output

- **`cmap`** *(optional)* — (str) - Matplotlib colormap applied to single-channel image previews and plate heatmaps. Perceptually uniform maps ('viridis', 'inferno', 'magma') preserve the relative visibility of intensity differences; 'gray' resembles the raw single-channel microscope image. Any registered matplotlib name is accepted, with an '_r' suffix to reverse it. Default 'inferno' for image plots and 'viridis' for plate heatmaps.
- **`qc_plot_max_panels`** *(optional)* — (int) - Largest number of wells drawn in the threshold-diagnostic figure, taken in sorted well order. It exists so a 384-well plate does not produce a 384-panel figure; the CSVs always carry every well regardless. Default 12.
- **`seed_wells_from_cells`** *(optional)* — (bool) - Read the cell table as well, so a well holding host cells but no parasites appears in the results with a zero denominator instead of vanishing from the plate entirely. Switch it off only when the database has no cell table. Default True.
- **`save`** *(optional)* — (bool or list of bool) - Whether to save masks to disk. Can be a list of three booleans for [cell, nucleus, pathogen] independently. Default varies by module -- False for most, True for the sequencing and regression paths.

#### Runtime & Reliability

- **`verbose`** *(optional)* — (bool) - Print the resolved settings table, channel and model choices per object type, row counts per table, and object counts after each filter. It only adds console output; enable it to identify which stage produced an unexpected object count. The default is True for mask, UMAP, screen analysis, barcode mapping and Cellpose training, and False for measure, plotting helpers and regression.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Assay Inputs
    # Required settings
    'src': 'path',
    # Optional settings
    'parasite_table': 'pathogen',
    'compartment': 'pathogen',

    # Channels & Intensity
    # Required settings
    'outside_channel': 1,
    'total_channel': 0,
    # Optional settings
    'intensity_statistic': 'auto',
    'background_correction': 'none',
    'min_total_intensity': None,

    # Thresholding
    # Optional settings
    'outside_threshold_method': 'otsu',
    'outside_threshold': None,
    'threshold_agreement_tolerance': 0.5,
    'threshold_sensitivity': 0.25,
    'bimodality_cutoff': 0.5555555555555556,
    'extracellular_class': 'attached',

    # Controls & Minimum Counts
    # Conditionally required settings
    'control_wells': None,
    # Optional settings
    'control_quantile': 0.99,
    'min_control_objects': 10,
    'min_objects_for_threshold': 10,
    'min_objects_for_bimodality': 30,
    'min_parasites_per_well': 50,
    'inflation_warn': 0.05,

    # Object Filtering
    # Optional settings
    'min_parasite_area': 0,
    'max_parasite_area': None,

    # Condition Metadata
    # Required settings
    'pathogen_types': ['pc'],
    'pathogen_plate_metadata': [['c1'], ['c2']],
    # Optional settings
    'cell_types': ['Hela'],
    'cell_plate_metadata': None,
    'treatments': None,
    'treatment_plate_metadata': None,
    'group_column': 'condition',
    'level': 'object',
    'change_plate': False,

    # Assay Output
    # Optional settings
    'cmap': 'viridis',
    'qc_plot_max_panels': 12,
    'seed_wells_from_cells': True,
    'save': True,

    # Runtime & Reliability
    # Optional settings
    'verbose': False,
}

In [ ]:
analyze_invasion(settings)

## Outputs and next steps

Per-object invasion labels and per-well or per-condition invasion estimates.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)